# 04 - Deep learning: MLP (ANN) and CNN
The deep models skip HOG: they learn their own features from the **RGB pixels** (64 x 64 x 3, scaled to 0-1).
They use the **same train/test split** as notebook 03.

| Model | Idea |
|---|---|
| MLP / ANN | flatten the image -> fully-connected layers |
| CNN | convolution filters learn local patterns (edges -> textures -> parts) |

In [ ]:
import sys; sys.path.append("..")          # lets the notebook import src/common.py
from src.common import *
import matplotlib.pyplot as plt, numpy as np, pandas as pd
print("Project root:", ROOT)
print("QUICK mode:", QUICK)
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)

X_train, X_test, y_train, y_test = load_split()
Xtr = X_train.astype("float32") / 255.0
Xte = X_test.astype("float32") / 255.0
print("train", Xtr.shape, "test", Xte.shape, "| epochs:", EPOCHS)

def callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)]

def compile_(model, lr=1e-3):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

## 1 - MLP / ANN
![MLP](../docs/model_steps/07_mlp_ann.png)

Flatten 64x64x3 = 12,288 numbers -> Dense 512 -> Dense 256 -> softmax(8), with dropout against over-fitting.
A MLP ignores the 2-D structure of the image, so it is usually clearly weaker than a CNN.

In [ ]:
mlp = keras.Sequential([
    layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    layers.Flatten(),
    layers.Dense(512, activation="relu"), layers.Dropout(0.3),
    layers.Dense(256, activation="relu"), layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation="softmax"),
], name="mlp")
compile_(mlp); mlp.summary()

In [ ]:
t = time.time()
h_mlp = mlp.fit(Xtr, y_train, validation_split=0.15, epochs=EPOCHS["mlp"],
                batch_size=64, callbacks=callbacks(), verbose=2)
tt = time.time() - t
plot_history(h_mlp, "MLP")

In [ ]:
y_pred = mlp.predict(Xte, verbose=0).argmax(1)
evaluate("MLP", "Deep Learning", y_test, y_pred, tt)
show_predictions(X_test, y_test, y_pred, "MLP")
mlp.save(MODELS_DIR / "mlp.keras")

## 2 - CNN
![CNN](../docs/model_steps/08_cnn.png)

Three blocks of *[Conv-BatchNorm-ReLU] x 2 -> MaxPool -> Dropout* with 32/64/128 filters, then global average pooling and a small classifier.
Light **data augmentation** (flip / rotation / zoom) is built into the model and is only active during training.

In [ ]:
def conv_block(x, f):
    for _ in range(2):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(0.25)(x)

inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = layers.RandomFlip("horizontal")(inp)
x = layers.RandomRotation(0.05)(x)
x = layers.RandomZoom(0.1)(x)
for f in (32, 64, 128):
    x = conv_block(x, f)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)
out = layers.Dense(len(CLASSES), activation="softmax")(x)
cnn = keras.Model(inp, out, name="cnn")
compile_(cnn); cnn.summary()

In [ ]:
t = time.time()
h_cnn = cnn.fit(Xtr, y_train, validation_split=0.15, epochs=EPOCHS["cnn"],
                batch_size=64, callbacks=callbacks(), verbose=2)
tt = time.time() - t
plot_history(h_cnn, "CNN")

In [ ]:
y_pred = cnn.predict(Xte, verbose=0).argmax(1)
evaluate("CNN", "Deep Learning", y_test, y_pred, tt)
show_predictions(X_test, y_test, y_pred, "CNN")
cnn.save(MODELS_DIR / "cnn.keras")

### What did the CNN learn? Feature maps of one test image

In [ ]:
convs = [l for l in cnn.layers if isinstance(l, layers.Conv2D)]
probe = keras.Model(cnn.input, [convs[0].output, convs[2].output, convs[4].output])
maps = probe(Xte[:1], training=False)
titles = ["Conv block 1 (32 filters, 64x64)", "Conv block 2 (64 filters, 32x32)", "Conv block 3 (128 filters, 16x16)"]

fig, axes = plt.subplots(4, 9, figsize=(14, 6.5))
for r in range(4):
    for c in range(9): axes[r, c].axis("off")
axes[0, 0].imshow(X_test[0]); axes[0, 0].set_title("input", fontsize=8)
for r, (fm, ttl) in enumerate(zip(maps, titles), start=1):
    fm = fm.numpy()[0]
    axes[r, 0].text(0, .5, ttl, fontsize=7, wrap=True)
    for c in range(8):
        axes[r, c + 1].imshow(fm[..., c], cmap="viridis")
plt.suptitle("CNN feature maps (first 8 channels of each block)"); plt.tight_layout()
save_fig("04_cnn_feature_maps.png"); plt.show()

## Quick comparison of the two networks
Their metrics are saved in `results/metrics/`; the full comparison with all ten models is built in notebook 05.

In [ ]:
df = compare(); display(df[df["family"] == "Deep Learning"].round(4))